# TP2 — Exercise 5

Run the cells from top to bottom. This notebook includes its prerequisite classes.

### Exercise 5: Template Method pipeline
`run()` fixes the order of the four steps. The CSV subclass supplies their implementations and combines the reader, missing-value strategy, and transformation factory. The demonstration saves a separate file. Normalization also scales the numeric ID column here because this general transformation applies to all numeric columns; in a real dataset, identifiers would usually be excluded.

In [1]:
import pandas as pd
from abc import ABC, abstractmethod
from functools import wraps
from time import perf_counter

#### Setup from earlier exercises

In [2]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        if self.data is None:
            self.read()
        print(self.data.head(n))

In [3]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.select_dtypes(include="number").columns:
            cleaned[column] = cleaned[column].fillna(cleaned[column].mean())
        return cleaned


class FillMode(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.columns:
            modes = cleaned[column].mode()
            if not modes.empty:
                cleaned[column] = cleaned[column].fillna(modes.iloc[0])
        return cleaned


class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [4]:
def log_action(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print(f"Starting {function.__name__}")
        try:
            result = function(*args, **kwargs)
        except Exception:
            print(f"Failed {function.__name__}")
            raise
        print(f"Finished {function.__name__}")
        return result
    return wrapper


def log_time(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return function(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            print(f"{function.__name__} took {elapsed:.6f} seconds")
    return wrapper


class LoggedDataCleaner(DataCleaner):
    @log_action
    @log_time
    def clean(self, df):
        return super().clean(df)

In [5]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):
    def apply(self, df):
        transformed = df.copy()
        for column in transformed.select_dtypes(include="number").columns:
            values = transformed[column]
            minimum = values.min()
            spread = values.max() - minimum
            if spread == 0:
                transformed[column] = values.where(values.isna(), 0.0)
            else:
                transformed[column] = (values - minimum) / spread
        return transformed


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().copy()


class StandardizeText(DataTransform):
    def apply(self, df):
        transformed = df.copy()
        for column in transformed.select_dtypes(include=["object", "string"]).columns:
            transformed[column] = transformed[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return transformed


class TransformFactory:
    @staticmethod
    def create(name):
        transformations = {
            "normalize": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = name.strip().lower()
        if key not in transformations:
            raise ValueError(f"Unknown transformation: {name}")
        return transformations[key]()

#### Exercise 5 implementation

In [6]:
class DataPipeline(ABC):
    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.data

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, strategy, transformations):
        self.reader = CSVReader(input_path)
        self.cleaner = LoggedDataCleaner(strategy)
        self.output_path = output_path
        self.transformations = transformations
        self.data = None

    def load(self):
        self.data = self.reader.read()

    def clean(self):
        self.data = self.cleaner.clean(self.data)

    def transform(self):
        for name in self.transformations:
            transformation = TransformFactory.create(name)
            self.data = transformation.apply(self.data)

    def save(self):
        self.data.to_csv(self.output_path, index=False)

In [7]:
pipeline = CSVDataPipeline(
    input_path="Data/sample_data (2).csv",
    output_path="Data/cleaned_data.csv",
    strategy=FillMean(),
    transformations=["standardize_text", "remove_duplicates", "normalize"],
)
final_data = pipeline.run()
print(final_data)
print("Saved to:", pipeline.output_path)

Starting clean
clean took 0.000453 seconds
Finished clean
         id     name      age  height_cm  weight_kg         city     score
0  0.000000    alice  0.25000   0.333333   0.433333     new york  0.588235
1  0.166667      bob  0.40625   0.800000   0.500000  los angeles  0.882353
2  0.333333  charlie  1.00000   0.533333   0.566667      chicago  0.647059
3  0.500000    david  0.40625   0.466667   0.833333      houston  0.000000
4  0.666667      eva  0.00000   0.000000   0.000000     new york  0.764706
5  0.833333    frank  0.37500   1.000000   1.000000  los angeles  0.647059
6  1.000000    grace  0.40625   0.133333   0.166667      chicago  1.000000
Saved to: Data/cleaned_data.csv


### Check the result
These checks cover this sample: mean filling removes its missing numeric values, normalization stays within 0–1, and the saved CSV matches the pipeline output.

In [8]:
assert not final_data.isna().any().any()
numeric_data = final_data.select_dtypes(include="number")
assert numeric_data.ge(0).all().all()
assert numeric_data.le(1).all().all()
pd.testing.assert_frame_equal(pd.read_csv("Data/cleaned_data.csv"), final_data)
print("Sample checks passed.")

Sample checks passed.
